# Package comparison and decision record

Comparison of the exact PyPI releases tested on 2026-11-03: `swing-analyzer` 0.1.2, `golftracker` 2.0, and `golftrainer` 1.0.2. “Yes” means observed in installed code or successful execution—not merely claimed by project documentation.

In [1]:
from pathlib import Path
import json, platform
import matplotlib.pyplot as plt, numpy as np, pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

In [2]:
rows = [
("Role","turnkey body-pose analyzer","video-to-frame-data producer","JSON-to-DataFrame adapter"),
("Pose model","MediaPipe Pose complexity 1","MediaPipe Pose complexity 1","none; consumes coordinates"),
("Landmarks retained","selected derived fields only","33 x/y/z/visibility per frame","33 x/y/z/visibility from input"),
("2D / 3D","2D image geometry; MediaPipe z discarded","normalized x/y plus relative MediaPipe z","passes x/y/z/visibility; tables mainly x/y"),
("Clubhead tracking","No","Manual points + straight-line interpolation","passes input clubhead points"),
("Shaft tracking","No; fixed 54°/manual-angle proxy","No; draws thumb-to-clubhead segment","No"),
("Ball tracking","No","No in v2 schema","No"),
("Swing events","Automatic wrist-height heuristics","sklearn pose classes or manual labels","passes stored labels; no detector"),
("Address / finish semantics","first / last tracked frame","classified/labelled Start / Finish","input labels"),
("Impact semantics","hands-low heuristic","not represented as a pose class","none"),
("Full frame trajectory","selected fields internally returned","Yes: all pose + club fields","Yes when valid producer JSON loads"),
("Visibility/confidence","not retained","MediaPipe visibility retained","visibility exposed in mp DataFrame"),
("Smoothing","3-frame phase; pose/tracer visual EMA/MA","optional path MA/spline utilities; not pipeline","none in wheel"),
("Angles / geometry","spine, line turns, arm and proxy checks","geometry utility module; no analysis report","none in wheel"),
("Tempo","top-to-impact and backswing ratio heuristics","No","No"),
("Automatic operation","Mostly, body only","MediaPipe yes; phases/club usually manual","yes only after compatible JSON exists"),
("Manual labelling","None required","GUI phase and club labels central to usable samples","none; relies on upstream labels"),
("Multi-swing comparison","summary diff + phase-warped video","No","No"),
("Structured export","Markdown; frame dict only via Python return","pickle + JSON (schema inconsistent)","DataFrames; no writer"),
("Annotated video","Yes, but codec can fail silently","draw methods; CLI missing","No"),
("Reusable Python API","Yes, simple functions","Importable library; broken workflow CLIs","small importable factory/dataclasses"),
("CLI","Working","broken: invalid entry point and omitted scripts","broken: omitted scripts"),
("Dependency status","Python 3.9-3.12; MP 0.10.14","MP 0.8.10 unavailable; protobuf 3.20 pin","minimal pandas/numpy"),
("Packaging/code quality","moderate; monolithic and unchecked writer","poor wheel/schema/per-frame model lifecycle","poor wheel completeness/schema validation"),
("Licence verified","MIT file in wheel","MIT file in wheel and source repo","MIT file in wheel and source repo"),
("Maintenance evidence","PyPI 2026-08-10","PyPI 2023-11-25; repo commit 2024-11-10","PyPI 2024-01-06; repo commit 2024-02-12"),
("Usefulness here","best executable reference and event baseline","best raw-data/club-label data model concept","useful only as a schema/table cautionary example"),
]
comparison=pd.DataFrame(rows,columns=["Feature","swing-analyzer","golftracker","golftrainer"])
pd.set_option("display.max_colwidth",120)
display(comparison)
comparison.to_csv(ROOT/"outputs"/"package_comparison.csv",index=False)

,Feature,swing-analyzer,golftracker,golftrainer
0,Role,turnkey body-pose analyzer,video-to-frame-data producer,JSON-to-DataFrame adapter
1,Pose model,MediaPipe Pose complexity 1,MediaPipe Pose complexity 1,none; consumes coordinates
2,Landmarks retained,selected derived fields only,33 x/y/z/visibility per frame,33 x/y/z/visibility from input
3,2D / 3D,2D image geometry; MediaPipe z discarded,normalized x/y plus relative MediaPipe z,passes x/y/z/visibility; tables mainly x/y
4,Clubhead tracking,No,Manual points + straight-line interpolation,passes input clubhead points
5,Shaft tracking,No; fixed 54°/manual-angle proxy,No; draws thumb-to-clubhead segment,No
6,Ball tracking,No,No in v2 schema,No
7,Swing events,Automatic wrist-height heuristics,sklearn pose classes or manual labels,passes stored labels; no detector
8,Address / finish semantics,first / last tracked frame,classified/labelled Start / Finish,input labels
9,Impact semantics,hands-low heuristic,not represented as a pose class,none


## Strongest implementation by area

In [3]:
winners=pd.DataFrame([
("Pose extraction","golftracker","Retains every MediaPipe landmark, z and visibility; swing-analyzer discards most raw data."),
("Phase detection","swing-analyzer","Only automatic temporal event heuristic that ran without an external model/manual labels; still not ground truth."),
("Club tracking","golftracker","Only package with club-head coordinates, but they are chiefly manual plus linear interpolation."),
("Trajectory storage","golftracker","Frame-aligned body, pose and club fields in pickle/JSON."),
("Geometry","swing-analyzer","Most integrated derived geometry and explicit camera-view suppression."),
("Visualization","swing-analyzer","Annotated video, key overlays, reports and synced comparison, subject to codec workaround."),
("Comparison","swing-analyzer","Only package with a two-swing path; it compares a small summary rather than distributions/consistency."),
],columns=["Area","Strongest","Why"])
display(winners)

,Area,Strongest,Why
0,Pose extraction,golftracker,"Retains every MediaPipe landmark, z and visibility; swing-analyzer discards most raw data."
1,Phase detection,swing-analyzer,Only automatic temporal event heuristic that ran without an external model/manual labels; still not ground truth.
2,Club tracking,golftracker,"Only package with club-head coordinates, but they are chiefly manual plus linear interpolation."
3,Trajectory storage,golftracker,"Frame-aligned body, pose and club fields in pickle/JSON."
4,Geometry,swing-analyzer,Most integrated derived geometry and explicit camera-view suppression.
5,Visualization,swing-analyzer,"Annotated video, key overlays, reports and synced comparison, subject to codec workaround."
6,Comparison,swing-analyzer,Only package with a two-swing path; it compares a small summary rather than distributions/consistency.


## Fit to the eventual project

None meets the central requirement. `golftracker` comes closest to the desired row model (`frame`, `timestamp`, all body landmarks, `club_head_x/y`, `phase`) but is manually labelled, packaging-broken, and lacks shaft/ball/robust events. `swing-analyzer` is the strongest currently executable implementation and a useful transparent baseline, but it retains too little raw data and observes no club. `golftrainer` 1.0.2 does not actually analyse.

For personal consistency, anomalous-swing detection, outcome linkage, and population/multi-session comparison, all three lack a dataset/session layer, stable schemas, quality flags, feature tables, alignment uncertainty, and statistical modelling.

## Recommended direction (design only)

- **Reuse concept directly:** golftracker's frame-aligned raw landmark + club provenance model; swing-analyzer's explicit phase trace and camera-view caveats.
- **Adapt:** swing-analyzer's event baseline, smoothing separation, diagnostics, and compare visualization; golftracker's manual correction workflow as an audit tool.
- **Replace:** pose lifecycle, dependency/environment management, club interpolation, phase classifier, JSON schema/versioning, codec handling, and all coaching thresholds.
- **Ignore:** claims of automatic ball/shaft tracking in old documentation, unintegrated Canny/Hough experiments, and golftrainer-as-analysis in the published wheel.

The proposed V0.1 architecture is documented in `docs/research/benchmarks/package-review.md`; it is intentionally not implemented here.

In [4]:
print("EXECUTED_OK: comparison notebook reached the final cell on",platform.python_version())

EXECUTED_OK: comparison notebook reached the final cell on 3.11.18
